In [2]:
import glob
import os

import pandas as pd

series = []

for path in sorted(glob.glob("*.csv")):
    name = os.path.basename(path)
    if name == "predictors_monthly.csv":
        continue
    df = pd.read_csv(path)
    df[df.columns[0]] = pd.to_datetime(df[df.columns[0]])
    df = df.set_index(df.columns[0]).sort_index()
    df.index.name = "date"
    df = df.dropna(how="all")

    if (df.index.day == 1).mean() > 0.9:
        # start-of-month stamp: lag one month (Mar 1 -> Feb 28)
        df.index = df.index - pd.Timedelta(days=1)
    elif df.index.to_series().diff().dt.days.median() < 5:
        # daily data: last observation of each month; drop the incomplete final month
        last = df.index.max()
        df = df.groupby(df.index.to_period("M")).last()
        if last != last + pd.offsets.MonthEnd(0):
            df = df.iloc[:-1]
        df.index = df.index.to_timestamp(how="end").normalize()
    # month-end / quarter-end stamps are already fine
    df.index = df.index + pd.offsets.MonthEnd(0)
    series.append(df)

predictors = pd.concat(series, axis=1).sort_index()
predictors.index.name = "date"
predictors.to_csv("predictors_monthly.csv")

print(predictors.shape)
print(predictors.apply(lambda s: s.first_valid_index()))
predictors.tail()

(1294, 16)
AAA                          1918-12-31
BAA                          1918-12-31
CPIAUCSL                     1946-12-31
DCOILWTICO                   1986-01-31
GDPC1                        1946-12-31
GS10                         1953-03-31
INDPRO                       1918-12-31
vwretd                       1926-01-31
vwretx                       1926-01-31
TB3MS                        1933-12-31
UNRATE                       1947-12-31
VIXCLS                       1990-01-31
c                            1952-03-31
w                            1952-03-31
y                            1952-03-31
cay=c-0.218 a-0.801y+0.441   1952-03-31
dtype: datetime64[ns]


,AAA,BAA,CPIAUCSL,DCOILWTICO,GDPC1,GS10,INDPRO,vwretd,vwretx,TB3MS,UNRATE,VIXCLS,c,w,y,cay=c-0.218 a-0.801y+0.441
date,,,,,,,,,,,,,,,,
2026-05-31,5.52,6.00,332.568,91.16,NaN,4.47,102.9610,NaN,NaN,3.66,4.2,15.32,NaN,NaN,NaN,NaN
2026-06-30,5.76,6.19,332.813,70.56,NaN,4.60,103.2006,NaN,NaN,3.73,4.1,16.45,NaN,NaN,NaN,NaN
2026-07-31,5.88,6.32,334.131,86.16,NaN,4.68,103.2664,NaN,NaN,3.72,4.1,15.99,NaN,NaN,NaN,NaN
2026-08-31,6.03,6.47,336.298,87.03,NaN,4.99,103.9506,NaN,NaN,3.94,4.2,14.92,NaN,NaN,NaN,NaN
2026-09-30,NaN,NaN,NaN,97.18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.34,NaN,NaN,NaN,NaN
